# Notebook 03: Parameter Sweeps and Calibration

**Prerequisite:** Notebooks 01 and 02  
**New topics:** 1D sweeps, 2D landscape plots, automated calibration against experimental data  
**Time to complete:** ~45 minutes

---

## Why calibrate?

The Lee model has two free parameters: **fc** (current fraction) and **fm** (mass fraction). These numbers are not directly measurable — they encode physics that the 0D model cannot resolve explicitly (Rayleigh-Taylor mixing, anomalous current leakage, electrode surface effects).

**Calibration** is the process of choosing fc and fm so that the model's I(t) prediction matches your laboratory's measured waveform. Once calibrated, you can trust the model's predictions for things you cannot measure directly — neutron yield, pinch density, implosion velocity.

This notebook shows three things:
1. How to sweep a single parameter (fill pressure → yield)
2. How to map the 2D landscape of (fm, fc) to visualize the parameter space
3. How to run automated calibration using a real device's experimental data

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker

from dpf.validation.lee_model_comparison import (
    LeeModel,
    estimate_neutron_yield_from_lee_result,
)
from dpf.validation.calibration import LeeModelCalibrator, CalibrationResult
from dpf.validation.experimental import DEVICES
from dpf.presets import get_preset

print("Imports OK.")

## Step 1: Set up the base device

We will use the **UNU-ICTP** device for most sweeps — it is the device most familiar to AAAPT students, runs fast in simulation, and has well-documented experimental data.

In [ ]:
preset = get_preset('unu_ictp')
cc = preset['circuit']
sp = preset['snowplow']

m_D2  = 6.687e-27
k_B   = 1.381e-23
rho0  = preset['rho0']
p_Pa  = rho0 * k_B * 300 / m_D2
p_torr_nominal = p_Pa / 133.322

base_params = {
    'C':              cc['C'],
    'V0':             cc['V0'],
    'L0':             cc['L0'],
    'R0':             cc['R0'],
    'anode_radius':   cc['anode_radius'],
    'cathode_radius': cc['cathode_radius'],
    'anode_length':   sp['anode_length'],
    'fill_pressure_torr': p_torr_nominal,
}

E_stored = 0.5 * cc['C'] * cc['V0']**2

print(f"UNU-ICTP device:")
print(f"  Stored energy   : {E_stored/1e3:.1f} kJ")
print(f"  Charge voltage  : {cc['V0']/1e3:.0f} kV")
print(f"  Nominal pressure: {p_torr_nominal:.2f} Torr")
print(f"  fc (preset)     : {sp['current_fraction']}")
print(f"  fm (preset)     : {sp['mass_fraction']}")

## Step 2: Sweep fill pressure and plot Yn vs pressure

This is the experiment every DPF lab runs first: vary the fill pressure and measure neutron yield at each setting. The result is a bell-shaped curve with a maximum at the **pressure optimum**.

The optimum exists because:
- **Low pressure:** Sheet is fast but sweeps little mass. Pinch is underdense, lower yield.
- **High pressure:** Sheet is slow and heavy. Current peaks before the pinch. Yield drops.
- **Optimum:** Sheet reaches the anode tip exactly when the circuit current is at its maximum.

In [ ]:
# Sweep fill pressure over a wide range
pressures_torr = np.linspace(0.5, 8.0, 20)

lee = LeeModel(
    current_fraction=sp['current_fraction'],
    mass_fraction=sp['mass_fraction'],
)

Yn_list     = []
Ipeak_list  = []
tpinch_list = []

for p in pressures_torr:
    params = dict(base_params)
    params['fill_pressure_torr'] = p
    try:
        r  = lee.run(device_params=params)
        yn = estimate_neutron_yield_from_lee_result(r)
    except Exception:
        yn = 0.0
        r  = None
    Yn_list.append(yn)
    Ipeak_list.append(r.peak_current / 1e3 if r else 0)
    tpinch_list.append(r.pinch_time * 1e6 if r else 0)

Yn_arr    = np.array(Yn_list)
Ipeak_arr = np.array(Ipeak_list)

# Find the optimum pressure
idx_opt    = np.argmax(Yn_arr)
p_opt      = pressures_torr[idx_opt]
Yn_opt     = Yn_arr[idx_opt]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 5))
fig.suptitle('UNU-ICTP — Parameter Sweep: Fill Pressure', fontsize=13, fontweight='bold')

# Neutron yield
ax1.semilogy(pressures_torr, Yn_arr, 'C0o-', markersize=6, linewidth=2)
ax1.axvline(p_opt, color='C3', linestyle='--', linewidth=1.5,
            label=f'Optimum: {p_opt:.1f} Torr')
ax1.axvline(p_torr_nominal, color='gray', linestyle=':', linewidth=1.2,
            label=f'Nominal: {p_torr_nominal:.1f} Torr')
ax1.set_xlabel('Fill pressure (Torr)', fontsize=11)
ax1.set_ylabel('Neutron yield Yn', fontsize=11)
ax1.set_title('Yield vs pressure (pressure optimum)')
ax1.legend(fontsize=10)
ax1.grid(True, alpha=0.3)

# Peak current vs pressure
ax2.plot(pressures_torr, Ipeak_arr, 'C1o-', markersize=6, linewidth=2)
ax2.axvline(p_opt, color='C3', linestyle='--', linewidth=1.5, label=f'Yield optimum')
ax2.set_xlabel('Fill pressure (Torr)', fontsize=11)
ax2.set_ylabel('Peak current (kA)', fontsize=11)
ax2.set_title('Peak current vs pressure')
ax2.legend(fontsize=10)
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print(f"Pressure optimum: {p_opt:.2f} Torr  (max Yn = {Yn_opt:.2e})")
print(f"Note: yield optimum is NOT at the same pressure as peak current.")
print(f"The optimum is a balance between implosion speed and plasma density.")

## Step 3: Sweep fm and plot I_peak vs fm

The mass fraction fm is not directly measurable in the lab, but its effect on I_peak is observable. By sweeping fm and comparing with a measured I_peak value, you can constrain the parameter.

In [ ]:
fm_values   = np.linspace(0.05, 0.40, 24)
fc_fixed    = sp['current_fraction']

Ipeak_fm  = []
tpinch_fm = []

for fm_val in fm_values:
    m = LeeModel(current_fraction=fc_fixed, mass_fraction=fm_val)
    r = m.run(device_params=base_params)
    Ipeak_fm.append(r.peak_current / 1e3)
    tpinch_fm.append(r.pinch_time * 1e6)

Ipeak_fm  = np.array(Ipeak_fm)
tpinch_fm = np.array(tpinch_fm)

# Published UNU-ICTP peak current (experimental)
exp_unu = DEVICES.get('UNU-ICTP')
Ipeak_exp_kA = exp_unu.peak_current / 1e3 if exp_unu else None

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 5))
fig.suptitle('Effect of fm (mass fraction) on UNU-ICTP Discharge', fontsize=13)

ax1.plot(fm_values, Ipeak_fm, 'C0o-', markersize=5, linewidth=2)
if Ipeak_exp_kA:
    ax1.axhline(Ipeak_exp_kA, color='C3', linestyle='--', linewidth=1.5,
                label=f'Experimental I_peak = {Ipeak_exp_kA:.0f} kA')
    # Find fm that matches experiment
    idx_match = np.argmin(np.abs(Ipeak_fm - Ipeak_exp_kA))
    ax1.axvline(fm_values[idx_match], color='C3', linestyle=':', linewidth=1,
                label=f'→ fm ≈ {fm_values[idx_match]:.2f}')

ax1.axvline(sp['mass_fraction'], color='gray', linestyle=':', linewidth=1,
            label=f'Preset fm = {sp["mass_fraction"]}')
ax1.set_xlabel('Mass fraction fm', fontsize=11)
ax1.set_ylabel('Peak current (kA)', fontsize=11)
ax1.set_title('I_peak vs fm  (fc fixed)')
ax1.legend(fontsize=9)
ax1.grid(True, alpha=0.3)

ax2.plot(fm_values, tpinch_fm, 'C4o-', markersize=5, linewidth=2)
ax2.set_xlabel('Mass fraction fm', fontsize=11)
ax2.set_ylabel('Pinch time (µs)', fontsize=11)
ax2.set_title('Pinch time vs fm  (heavier slug → slower)')
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("Key insight from the fm sweep:")
print("  fm is INVERSELY correlated with I_peak.")
print("  A heavy slug (high fm) takes longer to run down the electrode.")
print("  During that extra time, the capacitor discharges more → lower current at pinch.")
print()
print("  If your measured I_peak is lower than the simulation → fm is probably too small.")
print("  If your measured I_peak is higher → fm is probably too large.")

## Step 4: 2D landscape — how I_peak depends on (fc, fm) together

The two parameters fc and fm are **coupled**: changing one shifts the optimal value of the other. The right way to understand this is a 2D contour map — the "parameter landscape." You want to find the point in this landscape that matches your experimental measurement.

In [ ]:
# Define the grid — keep it moderate-resolution so it runs in a reasonable time
fc_grid = np.linspace(0.55, 0.85, 13)
fm_grid = np.linspace(0.05, 0.35, 13)

FC, FM = np.meshgrid(fc_grid, fm_grid)

# Allocate result arrays
Ipeak_grid  = np.zeros_like(FC)
Tpinch_grid = np.zeros_like(FC)
Yn_grid     = np.zeros_like(FC)

print(f"Running {FC.size} Lee model evaluations ({fc_grid.size} × {fm_grid.size} grid)...")

for i in range(fc_grid.size):
    for j in range(fm_grid.size):
        m = LeeModel(current_fraction=fc_grid[i], mass_fraction=fm_grid[j])
        try:
            r = m.run(device_params=base_params)
            Ipeak_grid[j, i]  = r.peak_current / 1e3
            Tpinch_grid[j, i] = r.pinch_time * 1e6
            Yn_grid[j, i]     = estimate_neutron_yield_from_lee_result(r)
        except Exception:
            Ipeak_grid[j, i]  = 0
            Tpinch_grid[j, i] = 0
            Yn_grid[j, i]     = 0

print("Done.")

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
fig.suptitle('UNU-ICTP — 2D Parameter Landscape (fc × fm)', fontsize=13, fontweight='bold')

# Helper: common plot settings
kw = dict(extent=[fc_grid.min(), fc_grid.max(), fm_grid.min(), fm_grid.max()],
          aspect='auto', origin='lower')

# --- I_peak ---
im1 = axes[0].imshow(Ipeak_grid, **kw, cmap='plasma')
cs1 = axes[0].contour(FC, FM, Ipeak_grid, colors='white', linewidths=0.8, alpha=0.7)
axes[0].clabel(cs1, fmt='%d kA', fontsize=7)
plt.colorbar(im1, ax=axes[0], label='I_peak (kA)')

# Mark experimental I_peak if available
if Ipeak_exp_kA:
    # Draw a contour line at the experimental value
    cs_exp = axes[0].contour(FC, FM, Ipeak_grid, levels=[Ipeak_exp_kA],
                              colors=['lime'], linewidths=2)
    axes[0].clabel(cs_exp, fmt=f'Exp: {Ipeak_exp_kA:.0f} kA', fontsize=8)

axes[0].set_xlabel('fc (current fraction)', fontsize=10)
axes[0].set_ylabel('fm (mass fraction)', fontsize=10)
axes[0].set_title('Peak current I_peak (kA)')

# --- Pinch time ---
im2 = axes[1].imshow(Tpinch_grid, **kw, cmap='viridis')
cs2 = axes[1].contour(FC, FM, Tpinch_grid, colors='white', linewidths=0.8, alpha=0.7)
axes[1].clabel(cs2, fmt='%.1f µs', fontsize=7)
plt.colorbar(im2, ax=axes[1], label='Pinch time (µs)')
axes[1].set_xlabel('fc (current fraction)', fontsize=10)
axes[1].set_ylabel('fm (mass fraction)', fontsize=10)
axes[1].set_title('Pinch time (µs)')

# --- Neutron yield (log scale) ---
Yn_log = np.log10(np.maximum(Yn_grid, 1))
im3 = axes[2].imshow(Yn_log, **kw, cmap='inferno')
plt.colorbar(im3, ax=axes[2], label='log₁₀(Yn)')
axes[2].set_xlabel('fc (current fraction)', fontsize=10)
axes[2].set_ylabel('fm (mass fraction)', fontsize=10)
axes[2].set_title('Neutron yield Yn (log₁₀)')

# Mark preset point on all panels
for ax in axes:
    ax.plot(sp['current_fraction'], sp['mass_fraction'], 'w*', markersize=12,
            label='Preset values')
    ax.legend(fontsize=8, loc='lower right')

plt.tight_layout()
plt.show()

print("Reading the landscape:")
print("  I_peak: increases with fc (more current drives the sheet), decreases with fm (heavier slug).")
print("  Pinch time: increases with fm (heavier slug → slower), weakly depends on fc.")
print("  The lime contour shows all (fc, fm) pairs that reproduce the experimental I_peak.")
print("  Calibration finds the point on that contour that also matches the TIMING.")

## Step 5: Why we need both I_peak AND timing to uniquely calibrate

From the landscape you can see that many (fc, fm) pairs give the same I_peak — they lie along a ridge. To uniquely determine both parameters, you need **two independent measurements**:

- I_peak constrains a combination of fc and fm
- The time at which I_peak occurs (or the pinch time) provides the second constraint

The calibrator minimizes a weighted error function:

> L(fc, fm) = w₁ · |I_peak_sim - I_peak_exp| / I_peak_exp  
>           + w₂ · |t_peak_sim - t_peak_exp| / t_peak_exp  
>           + w₃ · NRMSE(I(t) waveform)

Weights w₁ = 0.4, w₂ = 0.3, w₃ = 0.3 by default. The minimizer uses the Nelder-Mead simplex algorithm, which requires no gradients and handles the noisy ODE landscape well.

## Step 6: Run automated calibration for the UNU-ICTP device

The `LeeModelCalibrator` class wraps the Lee model, the experimental database, and the optimizer. All you need to provide is the device name.

In [ ]:
# Create the calibrator for UNU-ICTP
# The device database contains the experimental I_peak and waveform from published shots
calibrator = LeeModelCalibrator(
    device_name='UNU-ICTP',
    method='nelder-mead',
    peak_weight=0.4,
    timing_weight=0.3,
    waveform_weight=0.3,
)

print("Running calibration (Nelder-Mead optimizer)...")
print("Each optimizer iteration runs the Lee model ODE once.")
print()

# Run the calibration
# fc_bounds and fm_bounds define the allowed search region
cal_result = calibrator.calibrate(
    fc_bounds=(0.55, 0.85),
    fm_bounds=(0.05, 0.35),
    maxiter=80,
)

print(f"=== Calibration Result ===")
print(f"  Optimized fc           : {cal_result.best_fc:.4f}")
print(f"  Optimized fm           : {cal_result.best_fm:.4f}")
print(f"  Peak current error     : {cal_result.peak_current_error*100:.2f}%")
print(f"  Timing error           : {cal_result.timing_error*100:.2f}%")
print(f"  Optimizer converged?   : {cal_result.converged}")
print(f"  Number of evaluations  : {cal_result.n_evals}")

## Step 7: Compare before and after calibration

In [ ]:
exp_dev = DEVICES['UNU-ICTP']

# Before calibration: preset values
lee_before = LeeModel(
    current_fraction=sp['current_fraction'],
    mass_fraction=sp['mass_fraction'],
)
result_before = lee_before.run(device_params=base_params)

# After calibration: optimized values
lee_after = LeeModel(
    current_fraction=cal_result.best_fc,
    mass_fraction=cal_result.best_fm,
)
result_after = lee_after.run(device_params=base_params)

fig, ax = plt.subplots(figsize=(11, 5))

# Experimental waveform
if exp_dev.waveform_t is not None:
    ax.plot(exp_dev.waveform_t * 1e6, exp_dev.waveform_I / 1e3,
            'ko-', markersize=6, linewidth=1.5, label='Experimental (UNU-ICTP)', zorder=3)

# Before calibration
ax.plot(result_before.t * 1e6, result_before.I / 1e3,
        'C1--', linewidth=2,
        label=f'Before calibration  fc={sp["current_fraction"]:.2f}, fm={sp["mass_fraction"]:.2f}')

# After calibration
ax.plot(result_after.t * 1e6, result_after.I / 1e3,
        'C0', linewidth=2,
        label=f'After calibration   fc={cal_result.best_fc:.3f}, fm={cal_result.best_fm:.3f}')

ax.set_xlabel('Time (µs)', fontsize=12)
ax.set_ylabel('Current (kA)', fontsize=12)
ax.set_title('UNU-ICTP: Before vs After Calibration', fontsize=13)
ax.legend(fontsize=9)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# Print comparison table
Ipeak_exp = exp_dev.peak_current / 1e3
print(f"{'':30s} {'Before':>15} {'After':>15} {'Experimental':>15}")
print("-" * 78)
print(f"{'I_peak (kA)':<30} {result_before.peak_current/1e3:>15.1f} "
      f"{result_after.peak_current/1e3:>15.1f} {Ipeak_exp:>15.1f}")
print(f"{'Peak time (µs)':<30} {result_before.peak_current_time*1e6:>15.2f} "
      f"{result_after.peak_current_time*1e6:>15.2f} "
      f"{'N/A':>15}" if not exp_dev.current_rise_time else
      f"{exp_dev.current_rise_time*1e6:>15.2f}")
print(f"{'Error on I_peak':<30} "
      f"{abs(result_before.peak_current-exp_dev.peak_current)/exp_dev.peak_current*100:>14.1f}% "
      f"{cal_result.peak_current_error*100:>14.1f}%")

## Step 8: Use calibrated parameters for yield prediction

Once calibrated, you can use the model with confidence for predictions. Here we redo the pressure sweep from Step 2 with calibrated parameters and compare against the preset.

In [ ]:
# Pressure sweep with preset vs calibrated parameters
pressures = np.linspace(0.5, 8.0, 20)

Yn_preset = []
Yn_cal    = []

lee_preset = LeeModel(current_fraction=sp['current_fraction'], mass_fraction=sp['mass_fraction'])
lee_cal    = LeeModel(current_fraction=cal_result.best_fc, mass_fraction=cal_result.best_fm)

for p in pressures:
    params = dict(base_params)
    params['fill_pressure_torr'] = p
    try:
        r1 = lee_preset.run(device_params=params)
        Yn_preset.append(estimate_neutron_yield_from_lee_result(r1))
    except Exception:
        Yn_preset.append(0)
    try:
        r2 = lee_cal.run(device_params=params)
        Yn_cal.append(estimate_neutron_yield_from_lee_result(r2))
    except Exception:
        Yn_cal.append(0)

Yn_preset = np.array(Yn_preset)
Yn_cal    = np.array(Yn_cal)

fig, ax = plt.subplots(figsize=(10, 5))
ax.semilogy(pressures, Yn_preset, 'C1o--', markersize=5, linewidth=1.5,
            label=f'Preset fc={sp["current_fraction"]}, fm={sp["mass_fraction"]}')
ax.semilogy(pressures, Yn_cal, 'C0o-', markersize=5, linewidth=2,
            label=f'Calibrated fc={cal_result.best_fc:.3f}, fm={cal_result.best_fm:.3f}')

p_opt_preset = pressures[np.argmax(Yn_preset)]
p_opt_cal    = pressures[np.argmax(Yn_cal)]
ax.axvline(p_opt_preset, color='C1', linestyle=':', linewidth=1)
ax.axvline(p_opt_cal,    color='C0', linestyle=':', linewidth=1)

ax.set_xlabel('Fill pressure (Torr)', fontsize=11)
ax.set_ylabel('Neutron yield Yn', fontsize=11)
ax.set_title('Yield curve: preset vs calibrated parameters', fontsize=12)
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print(f"Pressure optimum:  preset = {p_opt_preset:.1f} Torr   calibrated = {p_opt_cal:.1f} Torr")
print(f"The calibrated model predicts a different optimal operating pressure.")
print(f"Trust the calibrated result for experimental planning.")

## Step 9: How to calibrate YOUR device

If you are working with a device that is not in the preset library, the workflow is:

1. **Measure I(t) on your device** using a Rogowski coil. Record several shots at the same pressure.
2. **Identify I_peak and t_peak** (time of current maximum) from the oscilloscope trace.
3. **Build a device_params dictionary** with your circuit parameters.
4. **Run a manual sweep** of fc and fm to bracket the experimental I_peak.
5. **Call the calibrator** (or iterate manually) to refine the values.

The cells below show a worked example using synthetic "experimental" data.

In [ ]:
# Example: calibrate a hypothetical 10 kJ device
# Imagine you measured I_peak = 250 kA at t_peak = 3.8 µs

# Your measured values
Ipeak_measured_kA = 250.0   # kA
tpeak_measured_us = 3.8     # µs

# Your device circuit parameters
my_device = {
    'C':              60e-6,      # 60 µF
    'V0':             20e3,       # 20 kV
    'L0':             80e-9,      # 80 nH
    'R0':             8e-3,       # 8 mΩ
    'anode_radius':   12e-3,      # 12 mm
    'cathode_radius': 35e-3,      # 35 mm
    'anode_length':   0.18,       # 18 cm
    'fill_pressure_torr': 3.5,    # 3.5 Torr D2
}

E_my = 0.5 * my_device['C'] * my_device['V0']**2
print(f"My device: {E_my/1e3:.0f} kJ stored at {my_device['V0']/1e3:.0f} kV")

# Sweep to find the (fc, fm) region that matches your measurement
fc_try = np.linspace(0.55, 0.85, 10)
fm_try = np.linspace(0.10, 0.40, 10)

best_error = 1e10
best_fc = sp['current_fraction']
best_fm = sp['mass_fraction']

for fc_val in fc_try:
    for fm_val in fm_try:
        m = LeeModel(current_fraction=fc_val, mass_fraction=fm_val)
        try:
            r = m.run(device_params=my_device)
            err_I = abs(r.peak_current/1e3 - Ipeak_measured_kA) / Ipeak_measured_kA
            err_t = abs(r.peak_current_time*1e6 - tpeak_measured_us) / tpeak_measured_us
            total_err = 0.5 * err_I + 0.5 * err_t
            if total_err < best_error:
                best_error = total_err
                best_fc = fc_val
                best_fm = fm_val
        except Exception:
            pass

print(f"\nManual grid search result:")
print(f"  Best fc   : {best_fc:.3f}")
print(f"  Best fm   : {best_fm:.3f}")
print(f"  Total error: {best_error*100:.1f}%")

# Verify
m_best = LeeModel(current_fraction=best_fc, mass_fraction=best_fm)
r_best = m_best.run(device_params=my_device)
print(f"\nAt best (fc, fm):")
print(f"  Simulated  I_peak = {r_best.peak_current/1e3:.1f} kA    (target: {Ipeak_measured_kA} kA)")
print(f"  Simulated  t_peak = {r_best.peak_current_time*1e6:.2f} µs  (target: {tpeak_measured_us} µs)")

## Step 10: Practical advice for calibration

Here is what experienced DPF users know from practice:

**Start with pressure.** Before tuning fc and fm, make sure your fill pressure is at or near the optimum. A large error in pressure will corrupt the fc/fm calibration.

**Use multiple shots.** A single oscilloscope trace has noise and shot-to-shot variation. Calibrate against the **average** of 5–10 reproducible shots at the same pressure. One-shot calibration is unreliable.

**Expect fc ≈ 0.6–0.8 and fm ≈ 0.05–0.25.** Values outside these ranges suggest a problem with the circuit parameters (check L₀ and R₀ carefully — these are often underestimated).

**fc and fm are device-specific and shot-independent.** Once calibrated at one pressure, they should work at other pressures on the same device (same electrodes, same gas species). If they do not, your circuit model may be wrong.

**Do not over-interpret.** The Lee model is a 0D approximation. Even with perfect calibration, it cannot predict shot-to-shot yield variation (which can be ±50% or more in DPF experiments due to kink instabilities and turbulence).

In [ ]:
# Summary: show calibrated yield curve for the UNU-ICTP over voltage range
voltages_kV = np.array([12, 14, 15, 16, 18, 20])
Yn_cal_V = []
Yn_pre_V = []

lee_cal2  = LeeModel(current_fraction=cal_result.best_fc, mass_fraction=cal_result.best_fm)
lee_pre2  = LeeModel(current_fraction=sp['current_fraction'], mass_fraction=sp['mass_fraction'])

for V0_kV in voltages_kV:
    params_v = dict(base_params)
    params_v['V0'] = V0_kV * 1e3
    try:
        r_c = lee_cal2.run(device_params=params_v)
        Yn_cal_V.append(estimate_neutron_yield_from_lee_result(r_c))
    except Exception:
        Yn_cal_V.append(0)
    try:
        r_p = lee_pre2.run(device_params=params_v)
        Yn_pre_V.append(estimate_neutron_yield_from_lee_result(r_p))
    except Exception:
        Yn_pre_V.append(0)

fig, ax = plt.subplots(figsize=(9, 5))
ax.semilogy(voltages_kV, Yn_cal_V, 'C0o-', markersize=8, linewidth=2,
            label='Calibrated model')
ax.semilogy(voltages_kV, Yn_pre_V, 'C1s--', markersize=8, linewidth=1.5,
            label='Preset model (uncalibrated)')
ax.set_xlabel('Charge voltage V₀ (kV)', fontsize=12)
ax.set_ylabel('Predicted neutron yield Yn', fontsize=12)
ax.set_title('UNU-ICTP: Calibrated vs Uncalibrated Yield Prediction', fontsize=12)
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print("The calibrated and uncalibrated models disagree on the absolute yield by a factor of ~2-5.")
print("The relative trend (Yn increases with voltage) is robust regardless of calibration.")
print("For planning experiments, always use calibrated parameters.")

## Summary

In this notebook you learned:

1. **1D pressure sweep:** Fill pressure has a bell-shaped yield optimum. The peak shifts with fc and fm.
2. **1D fm sweep:** Mass fraction is inversely correlated with I_peak and directly with pinch time.
3. **2D landscape:** The (fc, fm) parameter space shows clear structure. Many pairs reproduce I_peak, but only one reproduces both I_peak and timing.
4. **Why two measurements are needed:** I_peak alone cannot uniquely determine both fc and fm.
5. **Automated calibration:** `LeeModelCalibrator` runs Nelder-Mead optimization to minimize combined I_peak + timing + waveform error.
6. **Practical rules:** Calibrate against averaged shots at known pressure; fc ≈ 0.6–0.8, fm ≈ 0.05–0.25.

---

**Next steps:**
- Try running Notebook 04 to use the Metal GPU backend for full MHD simulations
- Apply these calibration techniques to your institution's DPF device
- Compare your calibrated fc/fm values against published Lee model results for similar devices